# REHABELT (PREDICTION MODEL)

###### FORCE INSTALL PYTHON 3.9

In [ ]:
%%bash
# Download Miniconda (Python 3.9 Installer)
MINICONDA_INSTALLER_SCRIPT=Miniconda3-py39_4.12.0-Linux-x86_64.sh
MINICONDA_PREFIX=/usr/local
wget https://repo.anaconda.com/miniconda/$MINICONDA_INSTALLER_SCRIPT

# Install over the current Python
chmod +x $MINICONDA_INSTALLER_SCRIPT
./$MINICONDA_INSTALLER_SCRIPT -b -f -p $MINICONDA_PREFIX

rm $MINICONDA_INSTALLER_SCRIPT

PREFIX=/usr/local
Unpacking payload ...
Solving environment: ...working... done

# All requested packages already installed.

installation finished.
    You currently have a PYTHONPATH environment variable set. This may cause
    unexpected behavior when running the Python interpreter in Miniconda3.
    For best results, please verify that your PYTHONPATH only points to
    directories of packages that are compatible with the Python interpreter
    in Miniconda3: /usr/local


--2026-05-04 13:03:54--  https://repo.anaconda.com/miniconda/Miniconda3-py39_4.12.0-Linux-x86_64.sh
Resolving repo.anaconda.com (repo.anaconda.com)... 104.16.191.158, 104.16.32.241, 2606:4700::6810:bf9e, ...
Connecting to repo.anaconda.com (repo.anaconda.com)|104.16.191.158|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 76607678 (73M) [application/x-sh]
Saving to: ‘Miniconda3-py39_4.12.0-Linux-x86_64.sh.1’

     0K .......... .......... .......... .......... ..........  0% 47.1M 2s
    50K .......... .......... .......... .......... ..........  0% 6.15M 7s
   100K .......... .......... .......... .......... ..........  0% 6.80M 8s
   150K .......... .......... .......... .......... ..........  0% 34.3M 7s
   200K .......... .......... .......... .......... ..........  0% 17.5M 6s
   250K .......... .......... .......... .......... ..........  0% 48.7M 5s
   300K .......... .......... .......... .......... ..........  0% 32.4M 5s
   350K .......... .......... 

##### DOWNGRADE TENSORFLOW (Fixes "Version 9" Crash)

In [ ]:
!pip uninstall -y tensorflow
!pip install tensorflow==2.10.0

Found existing installation: tensorflow 2.10.0
Uninstalling tensorflow-2.10.0:
  Successfully uninstalled tensorflow-2.10.0
  Using cached tensorflow-2.10.0-cp39-cp39-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (578.1 MB)


###### MOUNT GOOGLE DRIVE

In [ ]:
from google.colab import drive
import os

drive.mount('/content/drive', force_remount=True)

# Verify the main folder exists
folder_path = '/content/drive/MyDrive/rehabelt_data'
if os.path.exists(folder_path):
    print(f"\nSUCCESS: Found main folder: {folder_path}")
else:
    print(f"\nERROR: Main folder '{folder_path}' not found.")

Mounted at /content/drive

SUCCESS: Found main folder: /content/drive/MyDrive/rehabelt_data


##### MAIN SCRIPT

In [ ]:
%%bash

# SETUP
rm -rf legacy_env Miniconda3-py39_4.12.0-Linux-x86_64.sh
wget -q https://repo.anaconda.com/miniconda/Miniconda3-py39_4.12.0-Linux-x86_64.sh
chmod +x Miniconda3-py39_4.12.0-Linux-x86_64.sh
./Miniconda3-py39_4.12.0-Linux-x86_64.sh -b -f -p ./legacy_env > /dev/null
./legacy_env/bin/python -m pip install -q tensorflow==2.10.0 pandas openpyxl "numpy<2" scikit-learn matplotlib seaborn scipy

# MAIN SCRIPT
cat > builder_raw_aligned_charts.py << 'EOF'
import os
import glob
import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.model_selection import train_test_split
from tensorflow.keras.utils import to_categorical
from sklearn.utils import class_weight
from sklearn.preprocessing import StandardScaler
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns

RAW_BASE_DIR = '/content/drive/MyDrive/rehabelt_data'
OUTPUT_FOLDER = '/content/drive/MyDrive/Commission_Files'

TARGET_WINDOW_SIZE = 300
LEARNING_RATE = 0.0001
EPOCHS = 150
BATCH_SIZE = 32

os.makedirs(OUTPUT_FOLDER, exist_ok=True)

DIRS = {'raw_walk': 0, 'raw_sit_stand': 1, 'raw_supine_sit': 2}


def align_and_center_peak(data, label):
    if len(data) < 10:
        return None

    if label == 0:
        if len(data) >= TARGET_WINDOW_SIZE:
            start = (len(data) - TARGET_WINDOW_SIZE) // 2
            return data[start:start+TARGET_WINDOW_SIZE]
        else:
            pad_length = TARGET_WINDOW_SIZE - len(data)
            padding = np.repeat(data[-1:], pad_length, axis=0)
            return np.vstack((data, padding))

    else:
        accel_mag = np.sqrt(np.sum(data[:, :3]**2, axis=1))
        gyro_mag = np.sqrt(np.sum(data[:, 3:6]**2, axis=1))
        total_energy = accel_mag + gyro_mag
        peak_idx = np.argmax(total_energy)

        half_window = TARGET_WINDOW_SIZE // 2
        start_idx = peak_idx - half_window
        end_idx = peak_idx + half_window

        window = np.zeros((TARGET_WINDOW_SIZE, 6))

        data_start = max(0, start_idx)
        data_end = min(len(data), end_idx)
        window_start = max(0, -start_idx)
        window_end = window_start + (data_end - data_start)

        window[window_start:window_end] = data[data_start:data_end]

        if window_start > 0:
            window[:window_start] = data[0]
        if window_end < TARGET_WINDOW_SIZE:
            window[window_end:] = data[-1]

        return window

X_all, y_all = [], []

for folder_name, label in DIRS.items():
    folder_path = os.path.join(RAW_BASE_DIR, folder_name)
    if not os.path.exists(folder_path): continue
    files = glob.glob(os.path.join(folder_path, "*.[xX][lL][sS][xX]"))

    for file in files:
        try:
            df = pd.read_excel(file)
            data = df.iloc[:, 0:6].dropna().values

            window = align_and_center_peak(data, label)
            if window is not None:
                if label == 0 or np.sum(np.std(window, axis=0)) > 0.1:
                    X_all.append(window)
                    y_all.append(label)
        except: pass

X = np.array(X_all)
y = np.array(y_all)
print(f"Loaded {len(X)} perfectly centered windows. (Walk: {np.sum(y==0)}, Sit-Stand: {np.sum(y==1)}, Supine-Sit: {np.sum(y==2)})")

# NORMALIZE
scaler = StandardScaler()
X_flat = X.reshape(-1, 6)
X_scaled = scaler.fit_transform(X_flat).reshape(X.shape)

# UNRECOGNIZED CLASS
n_unrecognized = 150
X_unrecognized = []
for i in range(n_unrecognized):
    X_unrecognized.append(np.random.normal(0, 0.02, (TARGET_WINDOW_SIZE, 6)))

X_final = np.concatenate((X_scaled, np.array(X_unrecognized)), axis=0)
y_final = np.concatenate((y, np.full(n_unrecognized, 3)), axis=0)

# FLATTEN FOR DENSE
X_final = X_final.reshape(X_final.shape[0], -1)

# WEIGHTS
y_integers = y_final.astype(int)
class_weights = class_weight.compute_class_weight('balanced', classes=np.unique(y_integers), y=y_integers)
cw_dict = dict(enumerate(class_weights))

y_encoded = to_categorical(y_final, 4)
X_train, X_val, y_train, y_val = train_test_split(X_final, y_encoded, test_size=0.2, stratify=y_encoded)

# MODEL
model = Sequential([
    Dense(128, activation='relu', input_shape=(TARGET_WINDOW_SIZE * 6,)),
    Dropout(0.4),
    Dense(64, activation='relu'),
    Dropout(0.4),
    Dense(32, activation='relu'),
    Dense(4, activation='softmax')
])

model.compile(optimizer=Adam(learning_rate=LEARNING_RATE), loss='categorical_crossentropy', metrics=['accuracy'])

early_stop = EarlyStopping(monitor='val_loss', patience=12, restore_best_weights=True, verbose=1)

print("\nDense Model Training")
history = model.fit(X_train, y_train, epochs=EPOCHS, batch_size=BATCH_SIZE, validation_data=(X_val, y_val), verbose=1, class_weight=cw_dict, callbacks=[early_stop])

# EVALUATION
y_pred = np.argmax(model.predict(X_val), axis=1)
y_true = np.argmax(y_val, axis=1)
class_names = ['Walk', 'Sit-Stand', 'Supine-Sit', 'Unrecognized']

print("\n--- PERFORMANCE REPORT ---")
print(classification_report(y_true, y_pred, target_names=class_names))



# Confusion Matrix Plot
plt.figure(figsize=(8, 6))
cm = confusion_matrix(y_true, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=class_names, yticklabels=class_names)
plt.title('Validation Confusion Matrix')
plt.ylabel('True Class')
plt.xlabel('Predicted Class')
plt.tight_layout()
cm_path = os.path.join(OUTPUT_FOLDER, 'confusion_matrix.png')
plt.savefig(cm_path, dpi=300)
plt.close()

plt.figure(figsize=(12, 5))

# Accuracy Subplot
plt.subplot(1, 2, 1)
plt.plot(history.history['accuracy'], label='Train Accuracy', color='blue')
plt.plot(history.history['val_accuracy'], label='Val Accuracy', color='orange')
plt.title('Model Accuracy over Epochs')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()
plt.grid(True, linestyle='--', alpha=0.6)

# Loss Subplot
plt.subplot(1, 2, 2)
plt.plot(history.history['loss'], label='Train Loss', color='red')
plt.plot(history.history['val_loss'], label='Val Loss', color='orange')
plt.title('Model Loss over Epochs')
plt.xlabel('Epoch')
plt.ylabel('Loss (Categorical Crossentropy)')
plt.legend()
plt.grid(True, linestyle='--', alpha=0.6)

plt.tight_layout()
hist_path = os.path.join(OUTPUT_FOLDER, 'training_history.png')
plt.savefig(hist_path, dpi=300)
plt.close()

# EXPORT TFLITE
converter = tf.lite.TFLiteConverter.from_keras_model(model)
converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS]
tflite_model = converter.convert()

def hex_to_c_array(data):
    c_str  = '#include <cstdint>\n'
    c_str += f'const unsigned int model_data_len = {len(data)};\n'
    c_str += f'alignas(16) const unsigned char model_data[] = {{\n  '
    hex_array = [f'0x{val:02x}' for val in data]
    for i in range(0, len(hex_array), 12):
        c_str += ', '.join(hex_array[i:i+12])
        if i + 12 < len(hex_array): c_str += ",\n  "
    c_str += '\n};\n'
    return c_str

with open(os.path.join(OUTPUT_FOLDER, 'model_data.h'), "w") as f:
    f.write(hex_to_c_array(tflite_model))

print(f"MEAN: {scaler.mean_}")
print(f"SCALE: {scaler.scale_}")
EOF

# RUN
MPLBACKEND=Agg ./legacy_env/bin/python builder_raw_aligned_charts.py

Loaded 507 perfectly centered windows. (Walk: 111, Sit-Stand: 211, Supine-Sit: 185)

Dense Model Training
Epoch 1/150
17/17 [==============================] - 1s 16ms/step - loss: 1.5771 - accuracy: 0.3105 - val_loss: 1.1877 - val_accuracy: 0.4091
Epoch 2/150
17/17 [==============================] - 0s 5ms/step - loss: 1.1468 - accuracy: 0.4495 - val_loss: 1.0276 - val_accuracy: 0.5758
Epoch 3/150
17/17 [==============================] - 0s 5ms/step - loss: 0.9882 - accuracy: 0.5429 - val_loss: 0.8982 - val_accuracy: 0.6591
Epoch 4/150
17/17 [==============================] - 0s 4ms/step - loss: 0.8954 - accuracy: 0.6286 - val_loss: 0.8022 - val_accuracy: 0.8106
Epoch 5/150
17/17 [==============================] - 0s 4ms/step - loss: 0.7996 - accuracy: 0.6781 - val_loss: 0.7262 - val_accuracy: 0.8409
Epoch 6/150
17/17 [==============================] - 0s 5ms/step - loss: 0.7093 - accuracy: 0.7429 - val_loss: 0.6699 - val_accuracy: 0.8788
Epoch 7/150
17/17 [============================

2026-05-04 13:07:24.511052: I tensorflow/core/platform/cpu_feature_guard.cc:193] This TensorFlow binary is optimized with oneAPI Deep Neural Network Library (oneDNN) to use the following CPU instructions in performance-critical operations:  AVX2 AVX512F FMA
To enable them in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-05-04 13:07:24.682888: W tensorflow/stream_executor/platform/default/dso_loader.cc:64] Could not load dynamic library 'libcudart.so.11.0'; dlerror: libcudart.so.11.0: cannot open shared object file: No such file or directory; LD_LIBRARY_PATH: /usr/lib64-nvidia
2026-05-04 13:07:24.682933: I tensorflow/stream_executor/cuda/cudart_stub.cc:29] Ignore above cudart dlerror if you do not have a GPU set up on your machine.
2026-05-04 13:07:24.737966: E tensorflow/stream_executor/cuda/cuda_blas.cc:2981] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-05-04 13:07:25.7547